# vLLM tool-call parser probe — Colab/Kaggle

Runs a **tools-condition-only** PDDL-Copilot smoke matrix on a chosen Qwen-family model under both vLLM and Ollama, with timers and **tool-call extraction rate** as the headline metric. Pins each backend to its own GPU when 2 are visible (vLLM → GPU 1, Ollama → GPU 0); falls back to single-GPU sequential otherwise.

**Order: vLLM first, then Ollama.** vLLM is the migration candidate — running it first means a vLLM startup failure surfaces before we burn time on the Ollama baseline. The comparison cell is order-agnostic; output table keeps Ollama on the left, vLLM on the right regardless.

**Scope: tools only.** The no-tools condition is dropped — that path doesn't exercise the vLLM tool-call parser, which is the headline question this notebook is built to verify. (No-tools parity is already established under Ollama; see cluster-side production sweeps.) To re-enable, set `CONDITIONS = "both"`.

**Default vLLM tool-call parser: `qwen3_xml`.** Matches the Qwen3.5 / Qwen3.6 family `<tool_call><function=NAME><parameter=KEY>VAL</parameter></function></tool_call>` Llama-XML emit format. For vanilla Qwen3 (which emits Hermes JSON inside `<tool_call>`), set `TOOL_CALL_PARSER = "hermes"`.

**Companion** to `cluster-experimenting/run_smoke_vllm_vs_ollama.sbatch`. Knobs (MAX_MODEL_LEN / GPU_MEM_UTIL / ENFORCE_EAGER / NUM_PREDICT / TOOL_CALL_PARSER / HF_MODEL) are kept 1:1 with the sbatch so a Colab run reproduces the same vLLM serve config as the cluster job. Decision rule: `development/CHANGELOG.md` 2026-05-09 — migrate only if vLLM wall ≤ 0.7× Ollama AND tool-call extraction parity holds.


In [ ]:
# Parameters — edit and re-run.
MODEL_OLLAMA   = "Qwen3.5:0.8B"
MODEL_HF       = "Qwen/Qwen3.5-0.8B"
RUN_OLLAMA     = True                                                       # disable for vLLM-only run
RUN_VLLM       = True                                                       # disable for Ollama-only run
TASKS          = ["solve", "validate_domain", "validate_problem",
                  "validate_plan", "simulate"]                               # full 5-task matrix for parser-coverage
PARTIAL_K      = 1                                                          # 1 fixture per domain (fast slice)
NUM_VARIANTS   = 1
CONDITIONS     = "tools"                                                    # tools-only by default; set "both" to compare no-tools too
THINK          = "default"                                                  # default | on | off
CONCURRENCY    = 2

# vLLM serve knobs — kept 1:1 with cluster-experimenting/run_smoke_vllm_vs_ollama.sbatch
# so a Colab run reproduces the cluster job byte-for-byte (modulo dtype: T4 has no BF16).
MAX_MODEL_LEN     = 8192                  # T4 16 GB headroom; 16384 is too tight on dual-T4 with concurrency=2
GPU_MEM_UTIL      = 0.85
ENFORCE_EAGER     = True                  # Skip CUDA-graph capture (~30-60s startup); fine for smokes
NUM_PREDICT       = 4096                  # Override harness per-task num_predict (solve=8192 default would 400 against MAX_MODEL_LEN=8192). None = harness defaults.
TOOL_CALL_PARSER  = "qwen3_xml"           # qwen3_xml for Qwen3.5/3.6 (Llama-XML); hermes for vanilla Qwen3 (JSON-in-XML)
REASONING_PARSER  = "qwen3"

OLLAMA_GPU     = "0"                                                        # CUDA_VISIBLE_DEVICES for Ollama phase
VLLM_GPU       = "1"                                                        # CUDA_VISIBLE_DEVICES for vLLM phase
VLLM_PORT      = 8000
RESULTS_BASE   = ""                                                         # auto: /kaggle/working/results/vllm_smoke
RUN_TAG        = ""                                                         # auto-stamped if empty
EXPT_BRANCH    = "feat/vllm-smoke-probe"                                    # branch with --inference-backend

In [ ]:
import os, sys, time, subprocess, shlex, json, urllib.request
from pathlib import Path

PLATFORM = "kaggle" if Path("/kaggle").exists() else ("colab" if "google.colab" in sys.modules else "local")
print(f"Platform: {PLATFORM}")

WORK_DIR = "/kaggle/working/repos" if PLATFORM == "kaggle" else "/content"
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

if not RESULTS_BASE:
    RESULTS_BASE = "/kaggle/working/results/vllm_smoke" if PLATFORM == "kaggle" else str(Path(WORK_DIR) / "results")
if not RUN_TAG:
    RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = Path(RESULTS_BASE) / RUN_TAG
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print("OUTPUT_BASE:", OUTPUT_BASE)

EXPT_DIR        = Path(WORK_DIR) / "pddl-copilot-experiments"
MARKETPLACE_DIR = Path(WORK_DIR) / "pddl-copilot"

def sh(cmd, **kw):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=kw.pop("check", True), text=True, **kw).returncode

def _clone(url, dst, branch=None):
    if dst.exists():
        print(f"exists: {dst}")
        return
    branch_arg = f"--branch {shlex.quote(branch)} " if branch else ""
    sh(f"git clone --depth 1 {branch_arg}{url} {shlex.quote(str(dst))}")

_clone("https://github.com/SPL-BGU/pddl-copilot-experiments.git", EXPT_DIR, branch=EXPT_BRANCH)
_clone("https://github.com/SPL-BGU/pddl-copilot.git",            MARKETPLACE_DIR)

In [ ]:
# OS deps: Java for ENHSP, zstd for the Ollama installer, python3-venv for plugin venvs.
if PLATFORM in ("colab", "kaggle"):
    sh("apt-get -qq update")
    sh("apt-get -qq install -y openjdk-17-jre-headless python3-venv python3.12-venv zstd", check=False)
    sh("update-java-alternatives -s java-1.17.0-openjdk-amd64 2>/dev/null || true", check=False)
sh("java -version")

# Python deps (harness + openai client + vllm).
sh(f"pip install --quiet -r {shlex.quote(str(EXPT_DIR / 'requirements.txt'))}")
if RUN_VLLM:
    sh("pip install --quiet 'vllm>=0.6.0'")

In [ ]:
# Pre-create plugin venvs so the first MCP spawn is instant
# (mirrors cluster-experimenting/setup_env.sh).
import shutil
for plugin in ("pddl-solver", "pddl-validator"):
    plug_dir = MARKETPLACE_DIR / "plugins" / plugin
    venv = plug_dir / ".venv"
    pip  = venv / "bin" / "pip"
    if venv.exists() and not pip.exists():
        shutil.rmtree(venv)
    if pip.exists():
        print(f"{plugin}: venv ready")
        continue
    print(f"{plugin}: creating venv...")
    sh(f"python3 -m venv {shlex.quote(str(venv))}")
    sh(f"{shlex.quote(str(pip))} install --quiet --upgrade pip")
    sh(f"{shlex.quote(str(pip))} install --quiet -r {shlex.quote(str(plug_dir / 'requirements.txt'))}")

In [ ]:
# GPU detect — Kaggle "GPU T4 x2" gives two visible GPUs. Falls back gracefully.
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        text=True,
    )
    print(out)
    GPU_COUNT = len([l for l in out.strip().splitlines() if l])
except Exception as e:
    GPU_COUNT = 0
    print("nvidia-smi failed:", e)

print(f"GPU count: {GPU_COUNT}")
if RUN_OLLAMA and RUN_VLLM and GPU_COUNT < 2:
    print("WARNING: dual-T4 not detected — running both phases on GPU 0 sequentially.")
    print("         Numbers are still fair (each phase owns the GPU exclusively).")
    OLLAMA_GPU = VLLM_GPU = "0"

# Helpers shared by both phases.
def _run_smoke(cmd, cwd, log_path, label):
    """Stream `cmd` to the cell + tee to log_path; return (rc, wall_seconds)."""
    print(f"$ {' '.join(shlex.quote(c) for c in cmd)}")
    t0 = time.perf_counter()
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(log_path, "w") as logf:
        for line in proc.stdout:
            sys.stdout.write(line); logf.write(line)
    rc = proc.wait()
    wall = time.perf_counter() - t0
    print(f"\n[{label}] wall: {wall:.1f}s  rc={rc}")
    return rc, wall

def _stop(proc, name, timeout=15):
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            proc.kill()
    print(f"{name} stopped")

In [ ]:
# ====================  Phase 1: vLLM  ====================
VLLM_DIR = OUTPUT_BASE / "vllm"
VLLM_DIR.mkdir(parents=True, exist_ok=True)
vllm_wall = None

if RUN_VLLM:
    serve_env = {**os.environ, "CUDA_VISIBLE_DEVICES": VLLM_GPU}
    serve_log = VLLM_DIR / "vllm_serve.log"
    serve_cmd = [
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model",                   MODEL_HF,
        "--port",                    str(VLLM_PORT),
        "--host",                    "0.0.0.0",
        "--max-model-len",           str(MAX_MODEL_LEN),
        "--max-num-seqs",            "32",                # cap KV pre-alloc — vLLM v1 default 256 OOMs small VRAM
        "--enable-auto-tool-choice",
        "--tool-call-parser",        TOOL_CALL_PARSER,    # qwen3_xml for Qwen3.5/3.6; hermes for vanilla Qwen3
        "--reasoning-parser",        REASONING_PARSER,
        "--gpu-memory-utilization",  str(GPU_MEM_UTIL),
        "--dtype",                   "float16",           # T4 lacks BF16
    ]
    if ENFORCE_EAGER:
        serve_cmd.append("--enforce-eager")
    print("$ " + " ".join(shlex.quote(c) for c in serve_cmd))
    vllm_proc = subprocess.Popen(serve_cmd, stdout=open(serve_log, "w"),
                                 stderr=subprocess.STDOUT, env=serve_env)

    def _vllm_up():
        try: urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/v1/models", timeout=2); return True
        except Exception: return False

    print("Waiting for vLLM (up to 5 min for first run)...")
    ready = False
    for _ in range(150):
        if _vllm_up(): ready = True; break
        if vllm_proc.poll() is not None:
            print(f"vLLM died during startup; tail of {serve_log}:")
            sh(f"tail -50 {shlex.quote(str(serve_log))}", check=False)
            break
        time.sleep(2)
    if not ready:
        _stop(vllm_proc, "vllm")
        raise RuntimeError(f"vLLM did not become ready; see {serve_log}")
    print(f"vllm up on GPU {VLLM_GPU} (port {VLLM_PORT})")

    cmd = [
        "python3", "run_experiment.py",
        "--marketplace-path", str(MARKETPLACE_DIR),
        "--models",           MODEL_HF,
        "--inference-backend", "vllm",
        "--ollama-host",      f"http://localhost:{VLLM_PORT}",
        "--tasks",            *TASKS,
        "--conditions",       CONDITIONS,
        "--num-variants",     str(NUM_VARIANTS),
        "--concurrency",      str(CONCURRENCY),
        "--num-ctx",          str(MAX_MODEL_LEN),
        "--num-ctx-thinking", str(MAX_MODEL_LEN),
        "--partial",          str(PARTIAL_K),
        "--output-dir",       str(VLLM_DIR),
    ]
    if NUM_PREDICT is not None:
        cmd += ["--num-predict", str(NUM_PREDICT)]
    if THINK != "default":
        cmd += ["--think", THINK]

    rc, vllm_wall = _run_smoke(cmd, str(EXPT_DIR), VLLM_DIR / "run.log", "vLLM")
    _stop(vllm_proc, "vllm")
else:
    print("RUN_VLLM=False — skipping vLLM phase")

In [ ]:
# ====================  Phase 2: Ollama  ====================
OLLAMA_DIR = OUTPUT_BASE / "ollama"
OLLAMA_DIR.mkdir(parents=True, exist_ok=True)
ollama_wall = None

if RUN_OLLAMA:
    if subprocess.run("which ollama", shell=True, capture_output=True).returncode != 0:
        sh("curl -fsSL https://ollama.com/install.sh | sh")

    serve_env = {
        **os.environ,
        "OLLAMA_NUM_PARALLEL":   str(CONCURRENCY),
        "OLLAMA_KEEP_ALIVE":     "30m",
        "OLLAMA_CONTEXT_LENGTH": str(MAX_MODEL_LEN),
        "CUDA_VISIBLE_DEVICES":  OLLAMA_GPU,
    }
    serve_log = OLLAMA_DIR / "ollama_serve.log"
    ollama_proc = subprocess.Popen(["ollama", "serve"],
                                   stdout=open(serve_log, "w"),
                                   stderr=subprocess.STDOUT, env=serve_env)

    def _ollama_up():
        try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); return True
        except Exception: return False

    for _ in range(60):
        if _ollama_up(): break
        time.sleep(1)
    if not _ollama_up():
        _stop(ollama_proc, "ollama")
        raise RuntimeError(f"ollama serve did not become ready; see {serve_log}")
    print(f"ollama up on GPU {OLLAMA_GPU}")
    sh(f"ollama pull {shlex.quote(MODEL_OLLAMA)}")

    cmd = [
        "python3", "run_experiment.py",
        "--marketplace-path", str(MARKETPLACE_DIR),
        "--models",           MODEL_OLLAMA,
        "--inference-backend", "ollama",
        "--tasks",            *TASKS,
        "--conditions",       CONDITIONS,
        "--num-variants",     str(NUM_VARIANTS),
        "--concurrency",      str(CONCURRENCY),
        "--num-ctx",          str(MAX_MODEL_LEN),
        "--num-ctx-thinking", str(MAX_MODEL_LEN),
        "--partial",          str(PARTIAL_K),
        "--output-dir",       str(OLLAMA_DIR),
    ]
    if NUM_PREDICT is not None:
        cmd += ["--num-predict", str(NUM_PREDICT)]
    if THINK != "default":
        cmd += ["--think", THINK]

    rc, ollama_wall = _run_smoke(cmd, str(EXPT_DIR), OLLAMA_DIR / "run.log", "Ollama")
    _stop(ollama_proc, "ollama")
else:
    print("RUN_OLLAMA=False — skipping Ollama phase")

In [ ]:
# ====================  Comparison  ====================
def _read_trials(d):
    p = Path(d) / "trials.jsonl"
    if not p.exists(): return []
    return [json.loads(l) for l in p.open() if l.strip()]

def _stats(trials):
    if not trials: return None
    tools_trials = [t for t in trials if t.get("with_tools")]
    n = len(trials)
    n_tools = len(tools_trials)
    succ = sum(1 for t in trials if t.get("success"))
    extracted = sum(1 for t in tools_trials if (t.get("tool_calls") or []))
    selected  = sum(1 for t in tools_trials if t.get("tool_selected") is True)
    pt = sum((t.get("tokens") or {}).get("prompt", 0) for t in trials)
    ct = sum((t.get("tokens") or {}).get("completion", 0) for t in trials)
    et = sum((t.get("tokens") or {}).get("eval_duration_ns", 0) for t in trials) / 1e9
    return {
        "n": n, "n_tools": n_tools,
        "succ": succ, "rate": succ / n if n else 0.0,
        "extracted": extracted, "extracted_rate": (extracted / n_tools) if n_tools else 0.0,
        "selected":  selected,  "selected_rate":  (selected  / n_tools) if n_tools else 0.0,
        "prompt_tok": pt, "completion_tok": ct, "decode_s": et,
        "decode_tok_s": (ct / et) if et > 0 else 0.0,
    }

ollama_tr = _read_trials(OLLAMA_DIR) if RUN_OLLAMA else []
vllm_tr   = _read_trials(VLLM_DIR)   if RUN_VLLM   else []
o, v = _stats(ollama_tr), _stats(vllm_tr)

print()
print("=" * 78)
print(f"  vLLM tool-call parser probe — {MODEL_OLLAMA} / {MODEL_HF}")
print("=" * 78)
print(f"  Tasks:         {TASKS}")
print(f"  Conditions:    {CONDITIONS}    Concurrency: {CONCURRENCY}    MAX_MODEL_LEN: {MAX_MODEL_LEN}")
print(f"  vLLM parser:   {TOOL_CALL_PARSER}    reasoning: {REASONING_PARSER}    enforce_eager: {ENFORCE_EAGER}")
if NUM_PREDICT is not None:
    print(f"  num_predict:   {NUM_PREDICT}")
print()
print(f"  {'metric':28s} {'ollama':>14s} {'vllm':>14s} {'ratio (vllm/ollama)':>22s}")

def _row(name, oa, va, fmt="{:.1f}"):
    if oa is None or va is None or oa == 0:
        ratio = "n/a"
    else:
        ratio = f"{va/oa:.2f}x"
    oa_s = "n/a" if oa is None else fmt.format(oa)
    va_s = "n/a" if va is None else fmt.format(va)
    print(f"  {name:28s} {oa_s:>14s} {va_s:>14s} {ratio:>22s}")

_row("phase wall (s)",       ollama_wall,                    vllm_wall)
_row("trials run",           o["n"]              if o else None, v["n"]              if v else None, fmt="{:d}")
_row("tools trials",         o["n_tools"]        if o else None, v["n_tools"]        if v else None, fmt="{:d}")
_row("tool-extraction rate", o["extracted_rate"] if o else None, v["extracted_rate"] if v else None, fmt="{:.3f}")
_row("tool-selection rate",  o["selected_rate"]  if o else None, v["selected_rate"]  if v else None, fmt="{:.3f}")
_row("success rate",         o["rate"]           if o else None, v["rate"]           if v else None, fmt="{:.3f}")
_row("decode tok/s",         o["decode_tok_s"]   if o else None, v["decode_tok_s"]   if v else None, fmt="{:.1f}")
_row("prompt tokens",        o["prompt_tok"]     if o else None, v["prompt_tok"]     if v else None, fmt="{:d}")
_row("completion tokens",    o["completion_tok"] if o else None, v["completion_tok"] if v else None, fmt="{:d}")
_row("decode time (s)",      o["decode_s"]       if o else None, v["decode_s"]       if v else None)

# Per-task tool-extraction rate — surfaces which tasks the parser handles cleanly
# vs which ones get format anomalies (e.g. nested PDDL strings breaking XML parse).
def _per_task_extract(trials):
    out = {}
    for t in trials:
        if not t.get("with_tools"): continue
        task = t.get("task")
        out.setdefault(task, []).append(1 if (t.get("tool_calls") or []) else 0)
    return {k: (sum(v) / len(v), len(v)) for k, v in out.items()}

if vllm_tr:
    pt_v = _per_task_extract(vllm_tr)
    pt_o = _per_task_extract(ollama_tr) if ollama_tr else {}
    print()
    print("  Per-task tool-extraction rate (tools-condition only):")
    print(f"  {'task':18s} {'ollama (n)':>16s} {'vllm (n)':>16s}")
    for task in sorted(pt_v.keys() | pt_o.keys()):
        oa = pt_o.get(task, (None, 0))
        va = pt_v.get(task, (None, 0))
        oa_s = f"{oa[0]:.3f} ({oa[1]:d})" if oa[0] is not None else "n/a"
        va_s = f"{va[0]:.3f} ({va[1]:d})" if va[0] is not None else "n/a"
        print(f"  {str(task):18s} {oa_s:>16s} {va_s:>16s}")

# When vLLM extraction rate < 1.0 dump the first 3 raw responses where the
# parser gave up — fastest way to see whether the wrong parser was selected
# vs. some other format anomaly. Mirrors the diagnosis path used to find the
# original hermes/qwen3_xml mismatch.
if vllm_tr:
    failed = [t for t in vllm_tr if t.get("with_tools") and not (t.get("tool_calls") or [])][:3]
    if failed:
        print()
        print("  Sample vLLM responses where parser extracted no tool calls:")
        for i, t in enumerate(failed, 1):
            r = (t.get("response") or "")[:300]
            print(f"  [{i}] task={t.get('task')} done={t.get('done_reason')}  response[:300]:")
            for line in r.splitlines():
                print(f"      {line}")

print()
print("Outputs:")
if RUN_VLLM:   print(f"  vLLM:   {VLLM_DIR}")
if RUN_OLLAMA: print(f"  Ollama: {OLLAMA_DIR}")

## Notes

- **Headline metric: tool-extraction rate.** The fraction of tools-condition trials where vLLM's tool-call parser extracted any `tool_calls[]`. Independent of model correctness — a 0.0 here means the parser regex doesn't match the model's emit format (wrong parser flag), not that the model failed at the task. `tool-selection rate` is the stricter joint metric: extracted **and** the right tool was called.

- **Picking the right parser.** `qwen3_xml` for Qwen3.5 / Qwen3.6 family (emits `<tool_call><function=NAME><parameter=KEY>VAL</parameter></function></tool_call>` Llama-3 XML). `hermes` for vanilla Qwen3 (emits Hermes JSON inside `<tool_call>`). When `tool-extraction rate` is near 0, swap parsers and re-run — the "Sample vLLM responses where parser extracted no tool calls" debug block at the bottom of the comparison cell shows the actual emit format.

- **Order: vLLM first, then Ollama.** vLLM is the migration candidate; running it first means a startup failure surfaces immediately instead of after a 5-min Ollama baseline. The comparison cell is order-agnostic.

- **`--max-num-seqs 32`** (vs vLLM v1 default 256). Caps KV-cache pre-allocation; the default × `--max-model-len` × layers can OOM small VRAM during EngineCore init.

- **`MAX_MODEL_LEN` + `NUM_PREDICT` interplay.** vLLM rejects any request with `max_tokens > max_model_len` (HTTP 400). Harness per-task defaults are `solve=8192, validate_*=6144, simulate=6144`. With `MAX_MODEL_LEN=8192` (notebook default), set `NUM_PREDICT ≤ 4096` to leave prompt room. None = harness defaults (only safe if `MAX_MODEL_LEN ≥ 8192 + max_prompt_tokens`).

- **GPU pinning.** vLLM and Ollama are pinned to different GPUs via `CUDA_VISIBLE_DEVICES` so neither evicts the other. Single-GPU sessions auto-fall back to GPU 0 sequentially — still fair.

- **`--enforce-eager` on vLLM.** Skips CUDA graph compilation (~30–60s startup saving) at ~10–15% throughput cost. Fine for parser smokes; unset for production-style throughput numbers.

- **`--dtype float16`.** T4 lacks native BF16; vLLM auto-converts BF16 weights to FP16 on load. Numerically close enough for parser-format verification, but the cluster smoke (rtx_6000 / Ada) keeps native BF16 — those numbers are the authoritative throughput gate.

- **Decision rule.** `development/CHANGELOG.md` 2026-05-09: migrate only if vLLM wall ≤ 0.7× Ollama AND tool-call extraction parity holds. The notebook surfaces both directly.

- **Resume / re-run.** `run_experiment.py` reads `trials.jsonl` from each phase's output dir and skips completed trials, so re-executing a cell after a partial run only finishes what's missing. To start fresh, append `--no-resume` in the cmd list.

- **Re-enabling no-tools.** Set `CONDITIONS = "both"` to re-add the no-PDDL-tools cells. Useful for end-to-end parity sanity-checks; not required for parser verification.
